# Business Fraud & Anomaly Radar
### 2-Week MVP Prototype — Data & Analytics Suite
**Author**: Jephthah Kwame Lanor (Team Lead / Data & Analytics) & Data Team  
**Context**: AmaliTech Voluntary Internship  
**Project Objective**: Build an explainable, early-warning anomaly detection engine combining deterministic business rules and multi-dimensional machine learning (Isolation Forest) to surface high-risk SME transactions for human review.

---
### Notebook Roadmap:
1. **Environment Setup & Imports**
2. **Dataset Generation & Loading** (Realistic retail transactions + Injected fraud archetypes)
3. **Exploratory Data Analysis (EDA)** (Visualizing distributions, off-hours activity, and outlier patterns)
4. **Feature Engineering Pipeline** (Temporal features, employee baseline medians, deviation ratios)
5. **Deterministic Rule Engine (0–60 pts)** (Policy violations: duplicates, off-hours, discounts, refunds)
6. **Machine Learning Anomaly Detector (0–40 pts)** (Isolation Forest & feature-level explainability)
7. **Unified Hybrid Risk Scoring (0–100 pts)** (Calibration, severity classification, reason formatting)
8. **Evaluation & Ground-Truth Benchmark** (Detection rate across fraud archetypes)
9. **Export & Backend Contract Delivery** (Exporting demo alerts for Spring Boot & React dashboard)


## 1. Environment Setup & Library Imports


In [ ]:
import sys
import os
import datetime
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project analytics directory to sys.path so detection_engine can be imported directly
candidates = [
    os.path.abspath(os.path.join(os.getcwd(), '..')),
    os.path.abspath(os.path.join(os.getcwd(), '..', 'analytics')),
    os.path.abspath(os.getcwd())
]
for p in candidates:
    if p not in sys.path and os.path.exists(os.path.join(p, 'detection_engine')):
        sys.path.insert(0, p)

# Import our modular detection engine
from detection_engine import (
    AnomalyDetectionPipeline,
    FeatureEngineer,
    RuleEngine,
    MLAnomalyDetector,
    HybridScorer
)

# Plotting style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

print("Environment successfully initialized with pandas, scikit-learn, and custom detection engine!")


## 2. Ingesting Transaction Dataset
We load our synthetic SME dataset containing ~10,000 transactions across 8 employees and 3 branch stores, with ~3.5% injected anomalies representing known fraud patterns.


In [ ]:
# Find demo_transactions.csv in repository data/ or local data/
possible_paths = [
    os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'data', 'demo_transactions.csv')),
    os.path.abspath(os.path.join(os.getcwd(), '..', 'data', 'demo_transactions.csv')),
    os.path.abspath(os.path.join(os.getcwd(), 'data', 'demo_transactions.csv'))
]
csv_path = next((p for p in possible_paths if os.path.exists(p)), possible_paths[0])
print(f"Loading data from: {csv_path}")

# Load the generated dataset
df = pd.read_csv(csv_path)
print(f"Dataset Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
print("
First 5 Transactions:")
display(df.head(5))

print("\nGround Truth Injected Anomaly Breakdown:")
display(df['anomaly_type'].value_counts())


## 3. Exploratory Data Analysis (EDA)
Let's visualize the operational patterns to see how anomalies deviate from standard business hours and normal transaction amounts.


In [ ]:
# Extract temporary hour for visualization
df['temp_dt'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['temp_dt'].dt.hour

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1. Hourly Distribution
sns.histplot(
    data=df, 
    x='hour', 
    hue='anomaly_type', 
    multiple='stack', 
    bins=24, 
    discrete=True, 
    ax=axes[0]
)
axes[0].set_title("Transaction Volume by Hour of Day", fontsize=14, fontweight='bold')
axes[0].set_xlabel("Hour (24-Hour Format)")
axes[0].set_ylabel("Transaction Count")

# 2. Transaction Amount vs. Discount Percentage
sns.scatterplot(
    data=df, 
    x='amount', 
    y='discount_percent', 
    hue='anomaly_type', 
    alpha=0.7, 
    s=45, 
    ax=axes[1]
)
axes[1].set_title("Transaction Amount vs. Discount % (Spotting Outliers)", fontsize=14, fontweight='bold')
axes[1].set_xlabel("Amount ($)")
axes[1].set_ylabel("Discount (%)")

plt.tight_layout()
plt.show()


### Employee Spending & Anomaly Distribution
Notice how normal transactions cluster around each employee's typical ticket size, whereas injected amount spikes and refund frauds appear as sharp outliers.


In [ ]:
plt.figure(figsize=(14, 5))
sns.boxplot(
    data=df, 
    x='employee_id', 
    y='amount', 
    hue='is_anomaly', 
    palette={0: '#3498db', 1: '#e74c3c'},
    showfliers=True
)
plt.title("Transaction Amount Distribution by Employee (Normal vs Anomaly)", fontsize=14, fontweight='bold')
plt.xlabel("Employee ID")
plt.ylabel("Amount ($)")
plt.yscale('log')
plt.show()


## 4. Feature Engineering Pipeline
We extract temporal signals (`hour_of_day`, `is_off_hours`) and calculate historical employee baselines (median, deviation ratios).


In [ ]:
fe = FeatureEngineer(off_hours_start=21, off_hours_end=7)
fe.fit_profiles(df)
df_features = fe.transform(df)

print("Enriched Feature Columns:")
print(df_features[['transaction_id', 'amount', 'employee_median_amount', 'amount_to_median_ratio', 'discount_deviation', 'refund_ratio']].head(5))


## 5. Deterministic Rule Engine (0–60 Points)
The rule engine evaluates hard business constraints:
- Duplicate transactions within 10 minutes (+35 pts)
- Off-hours operations (22:00–06:00) (+25 pts)
- Excessive discount override (+25 pts)
- Unusual refund amount/ratio (+30 pts)
- Value spike vs employee median (+25 pts)
Scores are clamped to a maximum of 60 points.


In [ ]:
rule_engine = RuleEngine()
rule_scores, rule_reasons = rule_engine.evaluate(df_features)

df_features['rule_score'] = rule_scores
df_features['rule_reasons'] = rule_reasons

# Inspect triggered rules on anomalies
sample_rule_alerts = df_features[df_features['rule_score'] > 0][
    ['transaction_id', 'amount', 'anomaly_type', 'rule_score', 'rule_reasons']
].head(8)

display(sample_rule_alerts)


## 6. Machine Learning Anomaly Detector (0–40 Points)
Using `IsolationForest` with `RobustScaler` to detect subtle multi-dimensional outliers, combined with a Z-score explainability module.


In [ ]:
X_ml, feature_names = fe.get_ml_features(df_features)

ml_detector = MLAnomalyDetector(contamination=0.035)
ml_detector.fit(X_ml)
ml_scores, ml_reasons = ml_detector.score(X_ml)

df_features['ml_score'] = ml_scores
df_features['ml_reasons'] = ml_reasons

plt.figure(figsize=(10, 4))
sns.histplot(ml_scores, bins=25, kde=True, color='#8e44ad')
plt.title("Distribution of Calibrated ML Anomaly Scores (0 - 40 Scale)", fontsize=13, fontweight='bold')
plt.xlabel("ML Score")
plt.ylabel("Count")
plt.show()


## 7. Unified Hybrid Risk Scoring (0–100 Points)
We blend the scores using the formula:
$$\text{Risk Score} = \min(100, \text{Rule Score} + \text{ML Score})$$
And classify into four severity tiers:
- **LOW**: 0 – 29
- **MEDIUM**: 30 – 59
- **HIGH**: 60 – 79
- **CRITICAL**: 80 – 100


In [ ]:
scorer = HybridScorer()
final_results = scorer.combine(
    df=df_features,
    rule_scores=rule_scores,
    rule_reasons=rule_reasons,
    ml_scores=ml_scores,
    ml_reasons=ml_reasons
)

print("Severity Distribution:")
display(final_results['severity'].value_counts())

plt.figure(figsize=(8, 4))
severity_order = ['LOW', 'MEDIUM', 'HIGH', 'CRITICAL']
palette = {'LOW': '#2ecc71', 'MEDIUM': '#f39c12', 'HIGH': '#e67e22', 'CRITICAL': '#c0392b'}
sns.countplot(data=final_results, x='severity', order=severity_order, palette=palette)
plt.title("Alert Severity Breakdown", fontsize=14, fontweight='bold')
plt.xlabel("Severity Level")
plt.ylabel("Number of Transactions")
plt.show()


## 8. Evaluation Benchmark: Performance on Injected Frauds
Let's calculate the detection rate across all ground-truth anomaly archetypes.


In [ ]:
# Flagged as anomaly if severity is MEDIUM, HIGH, or CRITICAL (risk_score >= 30)
final_results['flagged_as_anomaly'] = final_results['risk_score'] >= 30

eval_df = final_results[final_results['anomaly_type'] != 'NONE']
recall_by_type = eval_df.groupby('anomaly_type')['flagged_as_anomaly'].mean() * 100

print("=== DETECTION RATE (RECALL) ON INJECTED ANOMALIES ===")
for anom_type, recall in recall_by_type.items():
    print(f"• {anom_type:<22}: {recall:.1f}% detected")

overall_recall = eval_df['flagged_as_anomaly'].mean() * 100
normal_fp_rate = final_results[final_results['anomaly_type'] == 'NONE']['flagged_as_anomaly'].mean() * 100

print("-" * 50)
print(f"Overall Anomaly Recall   : {overall_recall:.1f}%")
print(f"False Positive Rate (FP) : {normal_fp_rate:.2f}% (Normal transactions flagged)")


## 9. Sample High & Critical Alerts for Human Review
These alerts will be sent to the Spring Boot REST API and displayed on Opoku's React dashboard.


In [ ]:
top_alerts = final_results[final_results['severity'].isin(['HIGH', 'CRITICAL'])][
    ['transaction_id', 'timestamp', 'employee_id', 'amount', 'risk_score', 'severity', 'detector_type', 'reason_summary']
].head(10)

print(f"Displaying {len(top_alerts)} sample prioritized alerts:")
display(top_alerts)


## 10. Exporting Artifacts for Team Integration
We save the scored alerts to `data/scored_alerts.csv` and export sample JSON payloads for Albert (Backend) and Opoku (Frontend).


In [ ]:
data_dir = os.path.dirname(csv_path)
output_csv = os.path.join(data_dir, 'scored_alerts.csv')
final_results.to_csv(output_csv, index=False)
print(f"Full scored transactions exported to: {output_csv}")

# Export sample JSON payload for Backend API Contract
sample_payload = final_results[final_results['risk_score'] >= 60][
    ['transaction_id', 'employee_id', 'amount', 'risk_score', 'severity', 'detector_type', 'reasons']
].head(3).to_dict(orient='records')

json_output_path = os.path.join(data_dir, 'sample_alerts_payload.json')
with open(json_output_path, 'w') as f:
    json.dump(sample_payload, f, indent=2)

print(f"Sample JSON payload exported to: {json_output_path}")
print("\nSample JSON Contract Preview:")
print(json.dumps(sample_payload[0], indent=2))
